# **Pronóstico del precio de la energía en Colombia utilizando modelos de redes neuronales recurrenes**

##Por Alfredo Díaz

# Resumen del Proyecto

El objetivo de este proyecto es desarrollar un modelo predictivo basado en redes neuronales recurrentes (RNN) para predecir el precio de la energía en la bolsa de Colombia. La predicción del precio de la energía es un desafío debido a la variedad de factores que influyen en su formación, incluyendo la composición del parque generador (hidráulico, solar-eolica, térmico), eventos climáticos irregulares como El Niño y La Niña, los precios internacionales del petróleo y la correlación con la demanda de energía.

Para la construcción del modelo, se utilizarán datos provenientes de dos fuentes principales:

Yahoo Finance (yfinance): Para obtener información sobre los precios internacionales del petróleo.

XM - Sinergox: Plataforma que proporciona información sobre el Mercado de Energía Mayorista (MEM) y el Sistema Interconectado Nacional (SIN) en Colombia. A través de Sinergox, se accederá a datos clave como:

Informes periódicos sobre variables del MEM y SIN

Demanda, precios de bolsa, embalse y comportamiento del mercado

Datos principales de las variables del MEM y SIN

Alcance del Proyecto

El proyecto se centrará en los siguientes aspectos:

Recolección y preprocesamiento de datos: Obtención de información histórica de los precios del petróleo y las variables relevantes del mercado energético colombiano.

Desarrollo del modelo predictivo: Implementación de una red neuronal recurrente para la predicción del precio de la energía en la bolsa.

Evaluación del modelo: Validación del rendimiento del modelo mediante métricas de error y análisis de resultados.

Implementación en Google Colab: Se desarrollará un cuaderno de Jupyter en Google Colab que documente todo el proceso, desde la adquisición de datos hasta la generación de predicciones.

Aplicación en Streamlit para uso general.

A medida que se avanza en el desarrollo, se podrán incluir ajustes y mejoras en el modelo para optimizar la precisión de la predicción y su aplicabilidad en el mercado energético colombiano.



Importando bibliotecas
Las bibliotecas de Python nos facilitan el manejo de los datos y realizar tareas típicas y complejas con una sola línea de código.

Pandas – Esta biblioteca ayuda a cargar el marco de datos en un formato de matriz 2D y tiene múltiples funciones para realizar tareas de análisis de una sola vez.
Numpy Las matrices de numpy – son muy rápidas y pueden realizar grandes cálculos en muy poco tiempo.
Matplotlib/Adorno – Esta biblioteca se usa para dibujar visualizaciones.
Sklearn – Este módulo contiene múltiples bibliotecas que tienen funciones preimplementadas para realizar tareas desde el preprocesamiento de datos hasta el desarrollo y evaluación del modelo.
XGBoost – Contiene el algoritmo de aprendizaje automático eXtreme Gradient Boosting, que es uno de los algoritmos que nos ayuda a lograr una alta precisión en las predicciones.

In [ ]:
!pip install yfinance

!pip install xlwt

!pip install pydataxm



In [ ]:
#Para procesar datos, arreglos y dataframes

import numpy as np
import pandas as pd

import datetime as dt

#Se realiza la importación de las librerias necesarias para ejecutar
from pydataxm import *

#Para usar yfinance
import yfinance as yf

#Para graficar
import matplotlib.pyplot as plt
import seaborn as sb
import plotly.graph_objects as go

objetoAPI = pydataxm.ReadDB()                    #Se almacena el servicio en el nombre objetoAPI

# Obtención del Precio Internacional del Petróleo con yfinance

## Descripción de la actividad

En este script, utilizamos la librería `yfinance` para descargar los datos históricos del precio del petróleo crudo WTI desde el 1 de enero de 2020 hasta la fecha actual.

El objetivo es obtener y almacenar esta información para su posterior análisis en la predicción del precio de la energía en Colombia.




In [ ]:
import yfinance as yf
import pandas as pd
import datetime

# Definir el símbolo del petróleo WTI en Yahoo Finance
oil_ticker = 'CL=F'  # Futuros del crudo WTI

# Definir el rango de fechas
start_date = '2022-01-01'
end_date = datetime.datetime.today().strftime('%Y-%m-%d')

# Descargar los datos
oil_data = yf.download(oil_ticker, start=start_date, end=end_date)

# Mostrar las primeras filas del dataframe
oil_data[200:300]

In [ ]:
len(oil_data)

In [ ]:
oil_data.info()

In [ ]:
oil_data['Close'].plot(title="Precio al cierre del dia")

In [ ]:
oil_data.info()

In [ ]:
import plotly.graph_objects as go

title = "Movimiento del precio del petróleo"

# Corregir el acceso a las columnas MultiIndex
fig = go.Figure(data=[go.Candlestick(x=oil_data.index,
                                     open=oil_data[('Open', 'CL=F')],
                                     high=oil_data[('High', 'CL=F')],
                                     low=oil_data[('Low', 'CL=F')],
                                     close=oil_data[('Close', 'CL=F')])])

# Configuración del diseño del gráfico
fig.update_layout(title={"text": title},
                  xaxis={"rangeslider": {"visible": True}},
                  margin={"t": 30, "b": 0},
                  height=400,
                  width=1500)

fig.show()


In [ ]:
# Renombrar las columnas eliminando el MultiIndex
oil_data.columns = [col[0] for col in oil_data.columns]

# Verificar el cambio
print(oil_data.head())

#Precio de bolsa de energía en Colombia
Este código extrae datos del precio de bolsa de energía en Colombia utilizando una API.  
Los pasos realizados son los siguientes:

1. Se hace una solicitud a la API mediante `objetoAPI.request_data()`, especificando:
   - `"PrecBolsNaci"`: Identificador del precio de bolsa nacional.
   - `"Sistema"`: Nivel de agregación del dato (sistema eléctrico en su conjunto).
   - `dt.date(2022, 1, 1) y dt.date(2022, 1, 30)`: Rango de fechas para la extracción.

2. Se extraen los valores horarios del precio de bolsa de energía, que están en las columnas
   de la 2 a la 25 del DataFrame.

3. Se calcula el precio promedio diario de la bolsa de energía tomando el promedio
   de las 24 horas del día y se almacena en la nueva columna `'Daily_Average'`.

4. Finalmente, `df_pbolsa.head(5)` muestra las primeras 5 filas del DataFrame resultante
   para verificar la estructura de los datos extraídos.

Este procesamiento permite analizar la evolución diaria del precio de la energía en la bolsa de Colombia.


In [ ]:
df_pbolsa = objetoAPI.request_data(
                      "PrecBolsNaci",
                      "Sistema",
                      dt.date(2022, 1, 1),
                      dt.date.today())
names = [hour for hour in df_pbolsa.columns][2:26]
df_pbolsa['Daily_Average'] = df_pbolsa[names].mean(axis=1)

df_pbolsa.head(5)                           #Observar el encabezado del DataFrame extraido

In [ ]:
df_pbolsa.columns

In [ ]:
# Seleccionar las columnas de interés y renombrarlas
df_pbolsa_transformed = df_pbolsa.copy()

# Asignar 'Open' al valor de la primera hora (Values_Hour01)
df_pbolsa_transformed["Open"] = df_pbolsa["Values_Hour01"]

# Asignar 'Close' al valor de la última hora (Values_Hour24)
df_pbolsa_transformed["Close"] = df_pbolsa["Values_Hour24"]

# Calcular el máximo y mínimo de todas las horas para 'High' y 'Low'
hour_columns = [col for col in df_pbolsa.columns if "Values_Hour" in col]
df_pbolsa_transformed["High"] = df_pbolsa[hour_columns].max(axis=1)
df_pbolsa_transformed["Low"] = df_pbolsa[hour_columns].min(axis=1)

# Mantener solo las columnas necesarias y establecer 'Date' como índice
df_pbolsa_transformed = df_pbolsa_transformed[["Date", "Open", "High", "Low", "Close", "Daily_Average"]]
df_pbolsa_transformed.set_index("Date", inplace=True)

# Mostrar las primeras filas del DataFrame transformado
df_pbolsa_transformed.head()

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df_pbolsa_transformed["Daily_Average"])
plt.title('Precio de bolsa de energía en Colombia', fontsize=15,color='blue',family='serif')
plt.ylabel('Price in miles')
plt.show()

In [ ]:
import plotly.graph_objects as go

title = "Movimiento del precio en Bolsa"

# Corregir el acceso a las columnas MultiIndex
fig = go.Figure(data=[go.Candlestick(x=oil_data.index,
                                     open=df_pbolsa_transformed['Open'],
                                     high=df_pbolsa_transformed['High'],
                                     low=df_pbolsa_transformed['Low'],
                                     close=df_pbolsa_transformed['Close'])])

# Configuración del diseño del gráfico
fig.update_layout(title={"text": title},
                  xaxis={"rangeslider": {"visible": True}},
                  margin={"t": 30, "b": 0},
                  height=400,
                  width=1500)

fig.show()


# Demanda Comercial de Energia
Este código extrae datos de la demanda comercial de energía en Colombia utilizando la API de XM.  

Los pasos realizados son los siguientes:

1. Se hace una solicitud a la API mediante `objetoAPI.request_data()`, especificando:
   - `"DemaCome"`: Código del indicador de Demanda Comercial.
   - `"Sistema"`: Se solicita la demanda agregada a nivel del sistema eléctrico nacional.
   - `dt.date(2022, 1, 1)`: Fecha de inicio de los datos (1 de enero de 2022).
   - `dt.date(2025, 3, 7)`: Fecha de finalización (7 de marzo de 2025).

2. El resultado se almacena en `df_demanda`, un DataFrame de pandas que contendrá la información  
   de la demanda comercial de energía en el rango de fechas especificado.

Este dataset es útil para analizar el comportamiento del consumo de energía en Colombia,  
identificar patrones de demanda y posibles correlaciones con otras variables del mercado eléctrico.
"""


In [ ]:
df_demanda = objetoAPI.request_data(
                        "DemaCome",
                        "Sistema",
                        dt.date(2022,1,1),
                        dt.date.today())

In [ ]:
df_demanda

In [ ]:
df_demanda.columns

In [ ]:
# Copiar el DataFrame original
df_demanda_transformed = df_demanda.copy()

# Asignar 'Open' al valor de la primera hora (Values_Hour01)
df_demanda_transformed["Open"] = df_demanda["Values_Hour01"]

# Asignar 'Close' al valor de la última hora (Values_Hour24)
df_demanda_transformed["Close"] = df_demanda["Values_Hour24"]

# Identificar las columnas de horas
hour_columns = [col for col in df_demanda.columns if "Values_Hour" in col]

# Calcular el máximo y mínimo de todas las horas para 'High' y 'Low'
df_demanda_transformed["High"] = df_demanda[hour_columns].max(axis=1)
df_demanda_transformed["Low"] = df_demanda[hour_columns].min(axis=1)

# Calcular la demanda total sumando todas las horas
df_demanda_transformed["Hour_Average"] = df_demanda[hour_columns].mean(axis=1)

# Mantener solo las columnas necesarias y establecer 'Date' como índice
df_demanda_transformed = df_demanda_transformed[["Date", "Open", "High", "Low", "Close", "Hour_Average"]]
df_demanda_transformed.set_index("Date", inplace=True)

# Mostrar las primeras filas del DataFrame transformado
df_demanda_transformed.tail()

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df_demanda_transformed["Hour_Average"])
plt.title('Demanda Comercial de Energia', fontsize=15,color='blue',family='serif')
plt.ylabel('Price in miles')
plt.show()

In [ ]:
import plotly.graph_objects as go

title = "Demanda comercial de energía en Colombia"

# Corregir el acceso a las columnas MultiIndex
fig = go.Figure(data=[go.Candlestick(x=oil_data.index,
                                     open=df_demanda_transformed['Open'],
                                     high=df_demanda_transformed['High'],
                                     low=df_demanda_transformed['Low'],
                                     close=df_demanda_transformed['Close'])])

# Configuración del diseño del gráfico
fig.update_layout(title={"text": title},
                  xaxis={"rangeslider": {"visible": True}},
                  margin={"t": 30, "b": 0},
                  height=400,
                  width=1500)

fig.show()


# Precio de Escasez Marginal en el sistema energético colombiano
El siguiente código obtiene datos del Precio de Escasez Marginal en el sistema energético colombiano a través del API de XM.

Descripción de cada elemento:

1. ObjetoAPI.request_data(...)

Esta función realiza una solicitud de datos a la API de XM.
Extrae información en un rango de fechas determinado.

2. Parámetros de la solicitud:

"PrecEscaMarg": Código de la métrica "Precio de Escasez Marginal", el cual representa el precio umbral a partir del cual se activan mecanismos de seguridad energética según regulaciones.

"Sistema": Se indica que los datos serán a nivel del Sistema Interconectado Nacional (SIN) de Colombia.
dt.date(2022,1,1): Fecha de inicio de los datos (1 de enero de 2022).

dt.date.today(): Fecha de fin de los datos (fecha actual si se ajusta dinámicamente).

3. Resultado:

Se almacena la información en el DataFrame df_precioEscaMarg.
Este DataFrame contendrá la evolución del Precio de Escasez Marginal en el tiempo, permitiendo análisis de tendencias y su impacto en el mercado energético colombiano.


In [ ]:
df_precioEscaMarg = objetoAPI.request_data(
                        "PrecEscaMarg",
                        "Sistema",
                        dt.date(2022,1,1),
                        dt.date.today())

In [ ]:
df_precioEscaMarg

In [ ]:
df_precioEscaMarg.columns

In [ ]:
# Eliminar la columna "Id" si existe en el DataFrame
df_precioEscaMarg = df_precioEscaMarg.drop(columns=["Id"])

# Convertir la columna "Date" en el índice del DataFrame
df_precioEscaMarg.set_index("Date", inplace=True)

# Renombrar la columna "Value" para que represente las métricas OHLC
df_precioEscaMarg = df_precioEscaMarg.rename(columns={"Value": "Open"})

# Duplicar la misma columna para High, Low, Close y Average
df_precioEscaMarg["High"] = df_precioEscaMarg["Open"]
df_precioEscaMarg["Low"] = df_precioEscaMarg["Open"]
df_precioEscaMarg["Close"] = df_precioEscaMarg["Open"]
df_precioEscaMarg["Average"] = df_precioEscaMarg["Open"]

# Mostrar las primeras filas del DataFrame transformado
df_precioEscaMarg.head()


In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df_precioEscaMarg["Average"])
plt.title('Precio de Escasez Marginal ', fontsize=15,color='blue',family='serif')
plt.ylabel('Price in miles')
plt.show()

#  Precio de Escasez

Este fragmento de código utiliza la función `request_data` del objeto `objetoAPI` para obtener datos históricos del **Precio de Escasez** en el sistema eléctrico de Colombia.

**Parámetros utilizados:**
   - `"PrecEsca"`: Código del dato que representa el **Precio de Escasez**.
   - `"Sistema"`: Indica que se solicita información a nivel del sistema eléctrico nacional.
   - `dt.date(2022,1,1)`: Fecha de inicio de la consulta (1 de enero de 2022).
   - `dt.date.today()`: Fecha de fin de la consulta (hoy).

**Objetivo:**  
Obtener una serie de tiempo con el **Precio de Escasez** en el mercado mayorista de energía en Colombia para su posterior análisis y modelado.


In [ ]:
df_precioEsca = objetoAPI.request_data(
                        "PrecEsca",
                        "Sistema",
                        dt.date(2022,1,1),
                        dt.date.today())

In [ ]:
df_precioEsca

In [ ]:
# Eliminar la columna "Id" si existe en el DataFrame
df_precioEsca = df_precioEsca.drop(columns=["Id"])

# Convertir la columna "Date" en el índice del DataFrame
df_precioEsca.set_index("Date", inplace=True)

# Renombrar la columna "Value" para que represente las métricas OHLC
df_precioEsca = df_precioEsca.rename(columns={"Value": "Open"})

# Duplicar la misma columna para High, Low, Close y Average
df_precioEsca["High"] = df_precioEsca["Open"]
df_precioEsca["Low"] = df_precioEsca["Open"]
df_precioEsca["Close"] = df_precioEsca["Open"]
df_precioEsca["Average"] = df_precioEsca["Open"]

# Mostrar las primeras filas del DataFrame transformado
df_precioEsca.head()


In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df_precioEsca["Average"])
plt.title('Precio de Escasez', fontsize=15,color='blue',family='serif')
plt.ylabel('Price in miles')
plt.show()

#Demanda Real de energía

Este código obtiene datos históricos de la **Demanda Real de energía** en el sistema eléctrico de Colombia, utilizando la API de XM.

**Parámetros utilizados:**
   - `"DemaReal"`: Código del dato que representa la **Demanda Real de energía**.
   - `"Sistema"`: Se solicita la información a nivel del sistema interconectado nacional.
   - `dt.date(2022,1,1)`: Fecha de inicio de la consulta (1 de enero de 2022).
   - `dt.date.today()`: Fecha de fin de la consulta (7 de marzo de 2025).

**Objetivo:**  
Obtener una serie de tiempo con la demanda real de energía en Colombia, la cual representa el consumo total del Sistema Interconectado Nacional, sin incluir alumbrado público. Estos datos serán utilizados para análisis y modelado predictivo.



In [ ]:
df_DemaReal = objetoAPI.request_data(
                        "DemaReal",
                        "Sistema",
                        dt.date(2022,1,1),
                        dt.date.today())

In [ ]:
df_DemaReal

In [ ]:
# Copiar el DataFrame original
df_DemaReal_transformed = df_DemaReal.copy()

# Asignar 'Open' al valor de la primera hora (Values_Hour01)
df_DemaReal_transformed["Open"] = df_DemaReal["Values_Hour01"]

# Asignar 'Close' al valor de la última hora (Values_Hour24)
df_DemaReal_transformed["Close"] = df_DemaReal["Values_Hour24"]

# Identificar las columnas de horas
hour_columns = [col for col in df_DemaReal.columns if "Values_Hour" in col]

# Calcular el máximo y mínimo de todas las horas para 'High' y 'Low'
df_DemaReal_transformed["High"] = df_DemaReal[hour_columns].max(axis=1)
df_DemaReal_transformed["Low"] = df_DemaReal[hour_columns].min(axis=1)

# Calcular la demanda total sumando todas las horas
df_DemaReal_transformed["Hour_Average"] = df_DemaReal[hour_columns].mean(axis=1)

# Mantener solo las columnas necesarias y establecer 'Date' como índice
df_DemaReal_transformed = df_DemaReal_transformed[["Date", "Open", "High", "Low", "Close", "Hour_Average"]]
df_DemaReal_transformed.set_index("Date", inplace=True)

# Mostrar las primeras filas del DataFrame transformado
df_DemaReal_transformed[200:300]

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df_demanda_transformed["Hour_Average"])
plt.title('Demanda Real de Energia', fontsize=15,color='blue',family='serif')
plt.ylabel('Price in miles')
plt.show()

In [ ]:
df_pbolsa_transformed.columns

In [ ]:
df_precioEscaMarg.columns

In [ ]:
df_precioEsca.columns

In [ ]:
import matplotlib.pyplot as plt

# Crear la figura y los ejes
plt.figure(figsize=(12, 6))

# Graficar los datos de df_pbolsa_transformed
plt.plot(df_pbolsa_transformed.index, df_pbolsa_transformed["High"], label="Precio Bolsa - High", color="blue")
plt.plot(df_pbolsa_transformed.index, df_pbolsa_transformed["Low"], label="Precio Bolsa - Low", color="red")
plt.plot(df_pbolsa_transformed.index, df_pbolsa_transformed["Daily_Average"], label="Precio Bolsa - Daily Average", color="green")

# Graficar los datos de df_precioEscaMarg (Average)
plt.plot(df_precioEscaMarg.index, df_precioEscaMarg["Average"], label="Precio Escasez Marginal - Average", linestyle="dashed", color="purple")

# Graficar los datos de df_precioEsca (Average)
plt.plot(df_precioEsca.index, df_precioEsca["Average"], label="Precio Escasez - Average", linestyle="dashed", color="orange")

# Personalización del gráfico
plt.title("Comportamiento del Precio de Bolsa Nacional y Precio de Escasez")
plt.xlabel("Fecha")
plt.ylabel("Precio (COP/kWh)")
plt.legend()
plt.grid(True)

# Mostrar la gráfica
plt.show()


In [ ]:
import plotly.graph_objects as go

# Crear la figura
fig = go.Figure()

# Agregar las líneas al gráfico
fig.add_trace(go.Scatter(x=df_pbolsa_transformed.index, y=df_pbolsa_transformed["High"],
                         mode="lines", name="Precio Bolsa - High", line=dict(color="blue")))
fig.add_trace(go.Scatter(x=df_pbolsa_transformed.index, y=df_pbolsa_transformed["Low"],
                         mode="lines", name="Precio Bolsa - Low", line=dict(color="red")))
fig.add_trace(go.Scatter(x=df_pbolsa_transformed.index, y=df_pbolsa_transformed["Daily_Average"],
                         mode="lines", name="Precio Bolsa - Daily Average", line=dict(color="green")))
fig.add_trace(go.Scatter(x=df_precioEscaMarg.index, y=df_precioEscaMarg["Average"],
                         mode="lines", name="Precio Escasez Marginal - Average", line=dict(color="purple", dash="dash")))
fig.add_trace(go.Scatter(x=df_precioEsca.index, y=df_precioEsca["Average"],
                         mode="lines", name="Precio Escasez - Average", line=dict(color="orange", dash="dash")))

# Personalizar la gráfica
fig.update_layout(title="Comportamiento del Precio de Bolsa Nacional y Precio de Escasez",
                  xaxis_title="Fecha",
                  yaxis_title="Precio (COP/kWh)",
                  xaxis=dict(rangeslider=dict(visible=True), type="date"),
                  template="plotly_white")

# Mostrar la gráfica interactiva
fig.show()


#Capacidad Útil de Volumen en los Embalses

Estos datos del API corresponden a la Capacidad Útil de Volumen en los Embalses, tanto a nivel individual (por embalse) como agregado (por sistema).

 Descripción de los datos:

Capacidad Útil Volumen por Embalse (CapaUtilDiarMasa - Nivel Embalse): Representa el volumen útil disponible en cada embalse específico, medido en metros cúbicos (m³).
Capacidad Útil Volumen por Sistema (CapaUtilDiarMasa - Nivel Sistema): Es el volumen útil total de todos los embalses del sistema eléctrico nacional.

 Uso de estos datos:

Se utilizan para monitorear la disponibilidad de agua en los embalses, lo que es clave para la generación hidroeléctrica.
Permiten evaluar la resiliencia del sistema eléctrico ante sequías o eventos como El Niño y La Niña.

In [ ]:
df_CapaUtilDiarEner= objetoAPI.request_data(
                        "CapaUtilDiarEner",
                        "Sistema",
                        dt.date(2022,1,1),
                        dt.date.today())

In [ ]:
df_CapaUtilDiarEner

In [ ]:
# Agrupar por fecha y calcular los valores agregados
df_CapaUtilDiarEner_transformed = df_CapaUtilDiarEner.groupby("Date")["Value"].agg(
    Open="sum",    # Primer valor del día
    High="max",      # Máximo del día
    Low="min",       # Mínimo del día
    Close="sum",    # Último valor del día
    Average="mean"   # Promedio del día
).reset_index()

# Establecer 'Date' como índice
df_CapaUtilDiarEner_transformed.set_index("Date", inplace=True)

# Mostrar las primeras filas del DataFrame transformado
df_CapaUtilDiarEner_transformed.head()


In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df_CapaUtilDiarEner_transformed["Average"])
plt.title('Capacidad Útil de Volumen en los Embalses', fontsize=15,color='blue',family='serif')
plt.ylabel('Price in miles')
plt.show()

Este código consulta y extrae datos de Irradiación Global (IrrGlobal) desde el API, en un rango de fechas definido.

Explicación de los parámetros:

"IrrGlobal" → Hace referencia a la Irradiación Solar Global por recurso.
"Recurso" → Indica que los datos se obtienen a nivel de cada recurso de generación.
dt.date(2022,1,1) → Fecha de inicio: 1 de enero de 2022.
dt.date.today() → Fecha de fin: 7 de marzo de 2025.

¿Qué representa la Irradiación Global?
Es la cantidad total de radiación solar que incide sobre una superficie horizontal y se mide en vatios por metro cuadrado (W/m²).

Uso de estos datos:

Monitorear la disponibilidad de energía solar para generación fotovoltaica.
Analizar tendencias de radiación en distintos periodos.
Optimizar la planificación y operación de plantas solares.

In [ ]:
df_IrrGlobal= objetoAPI.request_data(
    "IrrGlobal",
    "Recurso",
    dt.date(2022,1,1),
    dt.date.today())

In [ ]:
df_IrrGlobal

In [ ]:
df_IrrGlobal.info()


In [ ]:
# Agrupar por fecha y calcular los valores agregados
df_IrrGlobal_transformed = df_IrrGlobal.groupby("Date")["Value"].agg(
    Open="sum",    # Primer valor del día
    High="max",      # Máximo del día
    Low="min",       # Mínimo del día
    Close="sum",    # Último valor del día
    Average="mean"   # Promedio del día
).reset_index()

# Establecer 'Date' como índice
df_IrrGlobal_transformed.set_index("Date", inplace=True)

# Mostrar las primeras filas del DataFrame transformado
df_IrrGlobal_transformed.head()


In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df_IrrGlobal_transformed["Average"])
plt.title('Promedio de Radiación Diaria en Colombia', fontsize=15,color='blue',family='serif')
plt.ylabel(' vatios por metro cuadrado (W/m²)')
plt.show()

# Temperatura ambiente solar por recurso en Colombia

DataFrame df_TempAmbSolar contiene los datos de temperatura ambiente solar por recurso en Colombia, obtenidos desde la API.

 Descripción de los datos:

Fuente: "TempAmbSolar" (Temperatura ambiente solar)
Nivel de agregación: "Recurso" (Datos reportados por cada recurso)
Periodo de consulta: Desde 2022-01-01 hasta 2025-03-07
Variables esperadas:
Id: Identificador del recurso
Code: Código del recurso
Value: Temperatura en grados Celsius (°C)
Date: Fecha de la medición


Posible análisis:

Calcular el promedio diario de temperatura ambiente para todos los recursos.
Identificar tendencias de temperatura en diferentes épocas del año.
Comparar con otros indicadores climáticos, como la radiación solar (IrrGlobal).

In [ ]:
df_TempAmbSolar=objetoAPI.request_data(
                        "TempAmbSolar",
                        "Recurso",
                        dt.date(2022,1,1),
                        dt.date.today())

In [ ]:
df_TempAmbSolar

In [ ]:
df_TempAmbSolar.info()

In [ ]:
# Agrupar por fecha y calcular los valores agregados
df_TempAmbSolar_transformed = df_TempAmbSolar.groupby("Date")["Value"].agg(
    Open="sum",    # Primer valor del día
    High="max",      # Máximo del día
    Low="min",       # Mínimo del día
    Close="sum",    # Último valor del día
    Average="mean"   # Promedio del día
).reset_index()

# Establecer 'Date' como índice
df_TempAmbSolar_transformed.set_index("Date", inplace=True)

# Mostrar las primeras filas del DataFrame transformado
df_TempAmbSolar_transformed.head()


In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df_TempAmbSolar_transformed["Average"])
plt.title('Promedio de Temperatura ambiente solar en Colombia', fontsize=15,color='blue',family='serif')
plt.ylabel('grados Celsius (°C)')
plt.show()

# **Análisis de datos exploratorios**
EDA es un enfoque para analizar los datos utilizando técnicas visuales. Se utiliza para descubrir tendencias y patrones, o para verificar supuestos con la ayuda de resúmenes estadísticos y representaciones gráficas.

Mientras realizamos el EDA de los datos del Precio promedio en Bolsa, vamos a integrar todos los dataframes cargado de las posibles variables, analizaremos cómo se han movido los precios de las acciones durante el período de tiempo y haremos los ajustes a los datos requeridos.

In [ ]:
!pip install workalendar

In [ ]:
from workalendar.america import Colombia
import pandas as pd
from datetime import datetime, timedelta

# Crear el calendario de Colombia
cal = Colombia()

# Generar un rango de fechas desde el 1 de enero de 2022 hasta hoy
fecha_inicio = datetime(2022, 1, 1)
fecha_fin = datetime.today()
fechas = pd.date_range(start=fecha_inicio, end=fecha_fin, freq="D")

# Obtener festivos año por año
festivos = []
for year in range(2022, fecha_fin.year + 1):
    festivos.extend(cal.holidays(year))  # Lista de tuplas (fecha, nombre del festivo)

# Convertir festivos a un conjunto de fechas para búsqueda rápida
festivos_set = {fecha for fecha, _ in festivos}

# Construir el DataFrame base
df_calendario = pd.DataFrame({"Date": fechas})

# Extraer Año, Mes y Día
df_calendario["Year"] = df_calendario["Date"].dt.year
df_calendario["Month"] = df_calendario["Date"].dt.month
df_calendario["Day"] = df_calendario["Date"].dt.day

# Identificar si es festivo
df_calendario["Holiday"] = df_calendario["Date"].isin(festivos_set)

# Identificar si es día hábil (lunes a viernes y no festivo)
df_calendario["Business_Day"] = df_calendario["Date"].dt.weekday < 5  # Lunes (0) a Viernes (4)
df_calendario["Business_Day"] = df_calendario["Business_Day"] & ~df_calendario["Holiday"]

# Establecer el índice en "Date"
df_calendario.set_index("Date", inplace=True)

# Mostrar las primeras filas
print(df_calendario.head())

In [ ]:
df_calendario.shape

In [ ]:
# Crear una copia del df_calendario para conservar el índice "Date"
df_final = df_calendario.copy()

# Hacer join con cada DataFrame, asegurando que la columna "Date" es el índice en cada uno
df_final = df_final.join(oil_data["Close"].rename("Precio_Oil"), how="left")
df_final = df_final.join(df_pbolsa_transformed["Daily_Average"].rename("Precio_bolsa"), how="left")
df_final = df_final.join(df_demanda_transformed["Hour_Average"].rename("Demanda_comercial"), how="left")
df_final = df_final.join(df_precioEsca["Average"].rename("Precio_escasez"), how="left")
df_final = df_final.join(df_DemaReal_transformed["Hour_Average"].rename("Demanda_real"), how="left")
df_final = df_final.join(df_CapaUtilDiarEner_transformed["Average"].rename("Capacidad_embalse"), how="left")
df_final = df_final.join(df_IrrGlobal_transformed["Average"].rename("Irradiacion"), how="left")
df_final = df_final.join(df_TempAmbSolar_transformed["Average"].rename("Temperatura"), how="left")

# Mostrar las primeras filas del DataFrame final
df_final.head()


In [ ]:
len(df_final)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Configurar el tamaño de la figura
plt.figure(figsize=(12, 6))

# Crear el mapa de calor de valores nulos
sns.heatmap(df_final.isna(), cmap="viridis", cbar=False, yticklabels=False)

# Agregar título
plt.title("Mapa de calor de valores nulos en df_final", fontsize=14)

# Mostrar la gráfica
plt.show()


In [ ]:
df_final.columns

In [ ]:
df_final.iloc[:, 5:] = df_final.iloc[:, 5:].fillna(method='ffill')
df_final.iloc[:, 5:] = df_final.iloc[:, 5:].fillna(method='bfill')

In [ ]:
df_final.isna().sum()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Definir columnas numéricas
numeric_columns = df_final.select_dtypes(include=['number']).columns

# Crear gráficos de boxplot por cada variable numérica
for column in numeric_columns:
    plt.figure(figsize=(4, 3))
    sns.boxplot(y=df_final[column])
    plt.title(f"Boxplot de {column}")
    plt.ylabel(column)
    plt.grid(True)
    plt.show()


In [ ]:
# Reemplazar temperaturas mayores a 50°C por 50
df_final.loc[df_final["Temperatura"] > 50, "Temperatura"] = 50

# Reemplazar radiación mayor a 1000 W/m² por 1000
df_final.loc[df_final["Irradiacion"] > 1000, "Irradiacion"] = 1000

# Verificar cambios
df_final[["Temperatura", "Irradiacion"]].describe()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Seleccionar las variables numéricas desde la posición 5 en adelante
df_corr = df_final.iloc[:, 5:].corr()

# Crear el mapa de calor
plt.figure(figsize=(10, 6))
sns.heatmap(df_corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)

# Configurar título
plt.title("Mapa de Calor de Correlación")

# Mostrar gráfico
plt.show()


In [ ]:
df_final.drop(columns=["Demanda_comercial"], inplace=True)

In [ ]:
df_final.info()

In [ ]:
df_final.describe()

In [ ]:
import numpy as np

# Normalizar el mes y el día en un ciclo anual
df_final["Month_sin"] = np.sin(2 * np.pi * df_final["Month"] / 12)
df_final["Month_cos"] = np.cos(2 * np.pi * df_final["Month"] / 12)
df_final["Day_sin"] = np.sin(2 * np.pi * df_final["Day"] / 31)
df_final["Day_cos"] = np.cos(2 * np.pi * df_final["Day"] / 31)

# Eliminar columnas innecesarias
df_final.drop(["Year", "Month", "Day"], axis=1, inplace=True)

In [ ]:
df_final.to_csv("pbolsa.csv", index=True)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

# Copia del dataframe para no modificar el original
df = df_final.copy()

# ============================
# 1 Normalización de datos
# ============================
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)

# ============================
# 2 Crear secuencias de tiempo
# ============================
def create_sequences(data, target_col, seq_length=30):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data.iloc[i:i+seq_length].values)  # Secuencia de 30 días
        y.append(data.iloc[i+seq_length][target_col])  # Día siguiente
    return np.array(X), np.array(y)

# Variable objetivo: Precio_bolsa
target_col = "Precio_bolsa"
seq_length = 30  # Usamos los últimos 30 días para predecir el siguiente

X, y = create_sequences(df_scaled, target_col, seq_length)

# ============================
# 3 Construir el modelo LSTM + MLP
# ============================
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(seq_length, df_scaled.shape[1])),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1)  # Predicción final
])

# ============================
# 4 Compilar y entrenar el modelo
# ============================
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

# Separar datos de entrenamiento y prueba (80% - 20%)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Entrenar el modelo
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50, batch_size=16)

# ============================
# 5 Evaluación y predicción
# ============================
import matplotlib.pyplot as plt

# Predicciones
y_pred = model.predict(X_test)

# Graficar resultados
plt.figure(figsize=(10, 5))
plt.plot(y_test, label="Real")
plt.plot(y_pred, label="Predicho")
plt.legend()
plt.title("Predicción de Precio de Bolsa")
plt.show()


# **División de datos y normalización**

In [ ]:
import numpy as np
import pandas as pd

# Crear nueva muestra con todas las columnas esperadas por el modelo (sin Year, Month, Day)
nueva_muestra = {
    "Business_Day": True,
    "Holiday": False,
    "Precio_Oil": 85.0,
    "Precio_bolsa": 0.0,  # Se agrega temporalmente, luego se predice
    "Precio_escasez": 800.0,
    "Demanda_real": 9.1e6,
    "Capacidad_embalse": 1.75e10,
    "Irradiacion": 420.0,
    "Temperatura": 28.0,
    "Month_sin": np.sin(2 * np.pi * 3 / 12),  # Marzo
    "Month_cos": np.cos(2 * np.pi * 3 / 12),
    "Day_sin": np.sin(2 * np.pi * 15 / 31),  # Día 15
    "Day_cos": np.cos(2 * np.pi * 15 / 31),
}

# Convertir a DataFrame
df_nueva_muestra = pd.DataFrame([nueva_muestra])

# Asegurar que las columnas coincidan con `df_final`
df_nueva_muestra = df_nueva_muestra[df_final.columns]  # Aquí ya no están 'Year', 'Month', 'Day'

# Aplicar la misma normalización
df_nueva_muestra_scaled = pd.DataFrame(scaler.transform(df_nueva_muestra), columns=df_scaled.columns)


In [ ]:
print(f"Columnas esperadas por el modelo: {df_scaled.shape[1]}")
print(f"Columnas en la muestra nueva: {df_nueva_muestra_scaled.shape[1]}")
print(f"Columnas en el modelo: {df_scaled.columns}")  # Las 13 que espera el modelo
print(f"Columnas en la nueva muestra: {df_nueva_muestra_scaled.columns}")  # Debe coincidir


In [ ]:
df_final.columns

In [ ]:
import numpy as np

# Reestructurar la muestra para que tenga la forma esperada por el modelo
X_nueva_muestra = np.expand_dims(df_nueva_muestra_scaled.values, axis=0)  # (1, num_features)

# Hacer la predicción con el modelo
precio_predicho_normalizado = model.predict(X_nueva_muestra )

# Crear un array temporal para desnormalizar correctamente
temp_array = np.zeros((1, df_scaled.shape[1]))  # Asegurar que coincida en tamaño

# Insertar la predicción en la posición de `Precio_bolsa`
temp_array[:, df_final.columns.get_loc("Precio_bolsa")] = precio_predicho_normalizado

# Desnormalizar la predicción
precio_predicho = scaler.inverse_transform(temp_array)[:, df_final.columns.get_loc("Precio_bolsa")]

# Mostrar el resultado
print(f" Predicción del Precio de Bolsa: {precio_predicho[0]:.2f}")


In [ ]:
df_scaled.shape[1]

In [ ]:
import joblib

# Suponiendo que tu normalizador se llama "scaler"
joblib.dump(scaler, "scaler.pkl")
print(" Normalizador guardado como 'scaler.pkl'")


In [ ]:
import tensorflow as tf

# Guardar el modelo en un directorio
model.save("pbolsa.keras")
print(" Modelo guardado en 'modelo_prediccion_precio'")
